In [39]:
import os
from langchain_ibm import ChatWatsonx
from langchain_ibm import WatsonxEmbeddings
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent
from dotenv import load_dotenv




In [40]:
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    max_tokens=2000,
    temperature=0
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)

### 1. 내장 Tool 사용

In [41]:
response = watson_llm.invoke("너는 몇년도 까지 학습되어있는 데이터를 가지고있나?")
response.content

"I am an AI language model developed by OpenAI, and I have been trained on a diverse range of internet text data up until 2021. However, I don't have real-time access to the internet or any private databases, so I can't provide information or answer questions about events, discoveries, or developments that have occurred after my training period."

In [42]:
from langchain_community.tools import DuckDuckGoSearchRun

google_search = DuckDuckGoSearchRun()

agent = create_agent(
    model=watson_llm, tools=[google_search], system_prompt="You are a helpful assistant"
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find out who the current president of South Korea is in 2026 using a web search",
            }
        ]
    }
)

result

{'messages': [HumanMessage(content='Find out who the current president of South Korea is in 2026 using a web search', additional_kwargs={}, response_metadata={}, id='7cd433cf-8954-4095-85d4-6354a0b0a5b5'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 237, 'total_tokens': 273}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-847cdf3f01cc0b86480b71d9792d65f6', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'current president of South Korea 2026'}, 'id': 'chatcmpl-tool-be26d855ec2dea8e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 237, 'output_tokens': 36, 'total_tokens': 273}),
  ToolMessage(content="2 days ago - Lee Jae Myung (Korean: 이재명; pronounced [i.dʑɛ.mjʌŋ]; born c. 8 December 1963) is a South Korean politician and lawyer who has served as the 14th president of South Korea since 2025. A member of 

In [43]:
def print_result(result):
    for i, msg in enumerate(result["messages"]):
        print(f"\n========= {i} ==========")
        print(type(msg).__name__)
        print(msg.content)

print_result(result)


========= 0 ==========
HumanMessage
Find out who the current president of South Korea is in 2026 using a web search

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
2 days ago - Lee Jae Myung (Korean: 이재명; pronounced [i.dʑɛ.mjʌŋ]; born c. 8 December 1963) is a South Korean politician and lawyer who has served as the 14th president of South Korea since 2025. A member of the Democratic Party of Korea (DPK), he was the party's leader while serving as member ... 5 days ago - Vice President of South Korea · ^ 본관; 本館; lit. ''Main House'' ^ 영빈관; 迎賓館; lit. ''Welcome-Guest House'' ^ As of 3 June 2026. ^ Goh Kun served as acting president during Roh's suspension from 12 March 2004 to 14 May 2004. ^ Hwang Kyo-ahn served as acting president during Park's suspension from 9 December 2016 to 10 May 2017. 5 days ago - Yoon Suk Yeol (Korean: 윤석열; pronounced [jun sʰʌŋnjʌɭ]; born 18 December 1960) is a South Korean politician who served as the 13th president of South Korea from 202

In [44]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search = GoogleSerperAPIWrapper()

google_search = Tool(
    name = "google_search",
    func=search.run,
    description="Google 검색으로 최신 정보를 찾습니다. 뉴스, 트렌드, 최신 기술 동향 검색에 사용하세요."
)

agent = create_agent(
    model=watson_llm, tools=[google_search], system_prompt="You are a helpful assistant"
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find out who the current president of South Korea is in 2026 using a web search",
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='Find out who the current president of South Korea is in 2026 using a web search', additional_kwargs={}, response_metadata={}, id='94b49e7f-7a01-4253-8581-b7dbdf9d1a0d'), AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 249, 'total_tokens': 285}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-71a57b3d0b19a6a04820990a9858e887', tool_calls=[{'name': 'google_search', 'args': {'__arg1': 'current president of South Korea 2026'}, 'id': 'chatcmpl-tool-b59fa658af256b4d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 249, 'output_tokens': 36, 'total_tokens': 285}), ToolMessage(content="Lee Jae Myung: President of South Korea. Lee Jae Myung is a South Korean politician and lawyer who has served as the 14th president of South Korea since 2025. A member of the Democratic Party of Korea, he was the p

In [45]:
# from langchain_community.utilities import GoogleSerperAPIWrapper, WikipediaAPIWrapper
# from langchain_community.tools import WikipediaQueryRun
# from langchain_core.tools import Tool


# search = WikipediaAPIWrapper(
#     user_agent="test/1.0",
#     lang="ko",
#     top_k_results=3,
#     doc_content_chars_max=2000
# )

# wiki_tool = WikipediaQueryRun(api_wrapper=search)

# agent = create_agent(
#     model=watson_llm, tools=[wiki_tool], system_prompt="You are a helpful assistant. Use the wikipedia tool to find facts and answer ONLY based on the tool results."
# )

# result = agent.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "Describe who is the Donald Trump",
#             }
#         ]
#     }
# )

# print(result)

In [46]:
import wikipediaapi
from langchain_core.tools import Tool

wiki_ko = wikipediaapi.Wikipedia(
    user_agent="test/1.0",
    language="ko"
)
wiki_en = wikipediaapi.Wikipedia(
    user_agent="test/1.0",
    language="en"
)


def wiki_search(query: str) -> str:
    for wiki in (wiki_ko, wiki_en):
        page = wiki.page(query)
        if page.exists():
            return page.summary[:2000]
    return f"'{query}' 에 해당하는 위키피디아 문서를 찾을 수 없습니다."


wiki_tool = Tool(
    name="wikipedia",
    func=wiki_search,
    description="위키피디아에서 인물, 사건, 개념 등 사실 정보를 찾습니다. 한국어와 영어 문서에서 검색합니다.",
)

agent = create_agent(
    model=watson_llm,
    tools=[wiki_tool],
    system_prompt="You are a helpful assistant. Use the wikipedia tool to find facts and answer ONLY based on the tool results.",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Describe who is Alan Turing",
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='Describe who is Alan Turing', additional_kwargs={}, response_metadata={}, id='3f694a6d-6f44-472e-99a6-9a93d8acdf32'), AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 257, 'total_tokens': 287}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-f1c5f69590d8a43240444025d967cb45', tool_calls=[{'name': 'wikipedia', 'args': {'__arg1': 'Alan Turing'}, 'id': 'chatcmpl-tool-8dd17db154c1c5c0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 257, 'output_tokens': 30, 'total_tokens': 287}), ToolMessage(content="Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorith

# 파일 관련 tool

In [47]:
from langchain_community.tools import ListDirectoryTool, ReadFileTool, WriteFileTool, CopyFileTool, DeleteFileTool

google_search = [
    ListDirectoryTool(),
    ReadFileTool(),
    WriteFileTool(),
    CopyFileTool(),
    DeleteFileTool(),
]

agent = create_agent(
    model=watson_llm,
    tools=google_search,
    system_prompt="You are a helpful assistant. Use the tools to find facts and answer ONLY based on the tool results.",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                # "content": "현재 파일 목록 보여줘",
                # "content" : "food.py 파일의 내용을 보여줘"
                # "content" : "현재 파일 목록을 summary.txt 파일 생성해서 저장해"
                # "content" : "summary.txt 파일 삭제해"
                "content" : "food.txt파일이 있다면 삭제하고, 그 다음 food.py의 내용을 읽고, food.py의 전체 file path를 기록하여 food.txt 파일을 만들어 저장해라"
            }
        ]
    }
)

def print_result(result):
    for i, msg in enumerate(result["messages"]):
        print(f"\n========= {i} ==========")
        print(type(msg).__name__)
        print(msg.content)

print_result(result)


========= 0 ==========
HumanMessage
food.txt파일이 있다면 삭제하고, 그 다음 food.py의 내용을 읽고, food.py의 전체 file path를 기록하여 food.txt 파일을 만들어 저장해라

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
.env
.python-version
.venv
agent.ipynb
agent2.ipynb
data
db
finder.py
food.py
food.txt
functiongemma-test.ipynb
langchain.ipynb
langchain2.ipynb
langchain3.ipynb
main.py
news.py
ollama_lesson.ipynb
ollama_lesson.js
output
pdf_rag.py
product.py
pyproject.toml
rag_company.py
rag_naver.ipynb
rag_분쟁조정.ipynb
README.md
research_report_20260605_131357.md
research_report_20260605_144156.md
research_report_20260605_144316.md
uv.lock
__pycache__

========= 3 ==========
AIMessage


========= 4 ==========
ToolMessage
File deleted successfully: food.txt.

========= 5 ==========
AIMessage


========= 6 ==========
ToolMessage
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOu

# 데이터베이스 agent

In [48]:
# db 생성

import sqlite3

conn = sqlite3.connect("./db/company.db")

cursor = conn.cursor()

cursor.execute("""DROP TABLE IF EXISTS employee""")

cursor.execute("""
               CREATE TABLE employee (id integer primary key, name text, salary integer)
               """)

cursor.execute("""
               INSERT into employee(name, salary)
               VALUES
               ('Kim', 4000),
               ('Lee', 5000),
               ('Park', 6000)
               """)

conn.commit()
conn.close()

In [49]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///./db/company.db")

print(db.run("SELECT * FROM employee"))


[(1, 'Kim', 4000), (2, 'Lee', 5000), (3, 'Park', 6000)]


In [50]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=watson_llm)
google_search = toolkit.get_tools()

agent = create_agent(
    model=watson_llm,
    tools=google_search,
    system_prompt="You are a database assistant",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "전체 직원 평균 급여는 얼마인가?",
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
전체 직원 평균 급여는 얼마인가?

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
employee

========= 3 ==========
AIMessage


========= 4 ==========
ToolMessage

CREATE TABLE employee (
	id INTEGER, 
	name TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employee table:
id	name	salary
1	Kim	4000
2	Lee	5000
3	Park	6000
*/

========= 5 ==========
AIMessage


========= 6 ==========
ToolMessage
[(5000.0,)]

========= 7 ==========
AIMessage
전체 직원의 평균 급여는 **5,000**입니다.


In [51]:
from langchain_classic.chains import LLMMathChain

LLMMathChain.from_llm(llm=watson_llm, verbose=True)
calc_tool = Tool(
    name="Calculator",
    func=LLMMathChain.from_llm(llm=watson_llm, verbose=True).run,
    description="수학 계산이 필요할 때 사용합니다. 시장 규모 계산, 성장률 계산, 단위 변환 등에 사용하세요."
)

calc_tool.invoke(
    "1200억 달러를 원화로 환산 , 1달러 = 1544.5원."
)



> Entering new LLMMathChain chain...
1200억 달러를 원화로 환산 , 1달러 = 1544.5원.Question: 1200억 달러를 원화로 환산 , 1달러 = 1544.5원.
```text
120000000000 * 1544.5
```
...numexpr.evaluate("120000000000 * 1544.5")...
```output
185340000000000.0
```
Answer: 185340000000000.0 원
> Finished chain.


'Answer:  185340000000000.0 원'

### 커스텀 Tool

- Description 작성
    기능설명, 언제 사용해야하는지, 입력값 형식, 출력값 형식, 필수 or 선택 명시

- 작성 방법
    - @tool 데코레이터
    - StructedTool
    - BaseTool

In [52]:
from langchain_core.tools import tool
import numexpr

# 함수 구현
@tool
def calculator(expression: str) -> str:
    """
    수식을 계산합니다.
    사칙연산에 사용하세요.
    """
    try:
        result = numexpr.evaluate(expression).item()
    except Exception as exc:
        return f"계산 오류: {exc}"
    return str(result)

calculator.invoke("123 * 567")

'69741'

In [53]:
agent = create_agent(
    model=watson_llm,
    tools=[calculator],
    system_prompt="You are a helpful assistant. Use the calculator tool to find facts and answer ONLY based on the tool results.",)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "1200억 달러(=120000000000달러)를 원화로 환산해 주세요. 1달러 = 1544.5원입니다."
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
1200억 달러(=120000000000달러)를 원화로 환산해 주세요. 1달러 = 1544.5원입니다.

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
185340000000000.0

========= 3 ==========
AIMessage
1200억 달러를 원화로 환산하면 **185,340조 원**입니다.


In [54]:
from pydantic import BaseModel, Field

class ExchangeRateInput(BaseModel):
    from_currency: str = Field(..., description="변환할 통화 코드 (예: USD, EUR, JPY, KRW)")
    to_currency: str = Field(
        ..., description="변환될 통화 코드 (예: USD, EUR, JPY, KRW)"
    )
    amount: float = Field(default=1.0, description="변환할 금액 (기본값:1)")

@tool(args_schema=ExchangeRateInput)
def get_exchange_rate(from_currency, to_currency, amount):
    """
    예제용 환율표를 조회하고 금액을 변환합니다.
    """
    mock_rates = {
        ('USD', 'KRW') : 1544.5,
        ('EUR', 'KRW') : 1440.0,
        ('JPY', 'KRW') : 9.2,
        ('KRW', 'USD') : 1 / 1544.5,
    }

    rate = mock_rates.get((from_currency, to_currency))

    if not rate:
        return "환율 정보 없음"
    converted = amount * rate

    return f"{amount:,.0f} {from_currency} = {converted:,.2f} {to_currency} (환율: {rate})"


agent = create_agent(model=watson_llm, tools = [get_exchange_rate])

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "1200억 달러(=120000000000달러)를 원화로 환산해 주세요. 1달러 = 1544.5원입니다."
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
1200억 달러(=120000000000달러)를 원화로 환산해 주세요. 1달러 = 1544.5원입니다.

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
120,000,000,000 USD = 185,340,000,000,000.00 KRW (환율: 1544.5)

========= 3 ==========
AIMessage
1200억 달러(=120,000,000,000 USD)를 원화로 환산하면 **185,340조 원**입니다.  

(환율: 1 USD = 1,544.5 KRW)


In [55]:
from langchain_core.tools import StructuredTool

def multiply(a:int, b:int) -> int:
    """두 숫자를 곱한다."""
    return a * b

multi_tool = StructuredTool.from_function(func=multiply)

agent = create_agent(model=watson_llm, tools=[multi_tool])
result = agent.invoke(
    {
        "messages":[
            {
                "role" : "user",
                "content": "12 곱하기 34는?"
            }
        ]
    }
)

print_result(result=result)


========= 0 ==========
HumanMessage
12 곱하기 34는?

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
408

========= 3 ==========
AIMessage
12를 34로 곱하면 **408**입니다.


In [56]:
def calculate_bmi(height: float, weight:float) -> str:
    """
    height(cm)와 weight(kg)를 입력받아 bmi 계산
    """
    bmi = weight / (((height) / 100) ** 2)
    return f"BMI = {bmi:.2f}"

In [57]:
bmi_tool = StructuredTool.from_function(func=calculate_bmi)

agent = create_agent(model=watson_llm, tools=[bmi_tool])
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "키 173cm, 몸무게 65kg 일때 BMI는?"
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
키 173cm, 몸무게 65kg 일때 BMI는?

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
BMI = 21.72

========= 3 ==========
AIMessage
키 173cm, 몸무게 65kg인 경우 BMI는 **21.72**입니다.


In [58]:
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city:str = Field(..., description="날씨를 조회할 도시명")

class WeatherTool(BaseTool):
    name:str = "weather"
    description:str = "특정 도시의 날씨를 조회합니다."

    args_schema:type[BaseModel] = WeatherInput

    def _run(self, city:str):
        
        weather_data = {
            "서울" : "맑음,30도",
            "부산" : "맑음,32도",
            "제주" : "비,20도",
        }

        return weather_data.get(city, f"{city} 날씨 정보 없음")

In [59]:
weather_tool = WeatherTool()

agent = create_agent(model=watson_llm, tools=[weather_tool])
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "서울의 날씨는?"
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
서울의 날씨는?

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
맑음,30도

========= 3 ==========
AIMessage
서울의 날씨는 맑고 기온은 30도입니다.


In [60]:
# Retriever Tool

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_classic.tools.retriever import create_retriever_tool

docs = [
    Document(page_content="연차는 입사 1년 미만은 월 1개 발생합니다."),
    Document(page_content="병가는 연 30일 까지 사용 가능합니다."),
    Document(page_content="퇴직금은 1년 이상 근무 시 지급됩니다."),
]

vectorstore = Chroma.from_documents(docs, embedding=watson_embedding)

retriever = vectorstore.as_retriever()

# retriever를 tool 등록
retriever_tool = create_retriever_tool(retriever, name="company_policy", description="회사 인사 규정을 검색합니다.")


In [61]:
agent = create_agent(
    model=watson_llm,
    tools=[retriever_tool],
    system_prompt="You are a helpful assistant. Use the company_policy tool to answer ONLY based on retrieved company policy.",
)

result = agent.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content" : "연차는 어떻게 발생하나요?"
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
연차는 어떻게 발생하나요?

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
연차는 입사 1년 미만은 월 1개 발생합니다.

연차는 입사 1년 미만은 월 1개 발생합니다.

퇴직금은 1년 이상 근무 시 지급됩니다.

퇴직금은 1년 이상 근무 시 지급됩니다.

========= 3 ==========
AIMessage
연차는 입사 후 1년 미만인 경우 **월 1일씩** 발생합니다.  
(1년 이상 근무 시 퇴직금도 지급됩니다.)


In [62]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=watson_llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Francisco?"}]}
)


{'messages': [HumanMessage(content='What is the weather in San Francisco?', additional_kwargs={}, response_metadata={}, id='2e49666b-d70d-4aa3-8f9f-eef51bbb41d2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 194, 'total_tokens': 222}, 'model_name': 'ibm/granite-4-h-small', 'system_fingerprint': '', 'finish_reason': 'tool_calls'}, id='chatcmpl-1fd76be65cc727f39669dd9c632fe377', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'chatcmpl-tool-a702d59e2112376a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 194, 'output_tokens': 28, 'total_tokens': 222}),
  ToolMessage(content="It's always sunny in San Francisco!", name='get_weather', id='ce5e59ee-52a0-4024-b015-0c820e52fb1d', tool_call_id='chatcmpl-tool-a702d59e2112376a'),
  AIMessage(content="It's always sunny in San Francisco!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tok

# 리서치 자동화 Agent 구현 
- 입력 : 리서치 주제
- tool : 
    - 1. 리서치 주제에 대해 웹으로 최신 동향 수집
    - 2. 최신 동향 요약
    - 3. 수치 데이터가 있으면 계산기로 검증/변환
    - 4. 마크다운 형식의 문서로 저장

In [63]:
# 1. 웹 검색 (google) tool

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool, tool

search = GoogleSerperAPIWrapper()
search_tool = Tool(
    name="google_search",
    func=search.run,
    description="Google 검색으로 최신 정보를 찾습니다. 뉴스, 트렌드, 최신 기술 동향 검색에 사용하세요."
)

# search_tool.run("AI 반도체 시장 2026년 전망")

In [64]:
@tool
def trend_analysis_tool(text):
    """검색 결과 텍스트를 입력받아 핵심 트렌드, 주요 기업, 성장 요인, 위험 요소를 분석합니다."""

    prompt = f"""
    아래 검색 결과를 분석하여

    1. 핵심 트렌드
    2. 주요 기업
    3. 성장 요인
    4. 위험 요소
    를 정리하세요.

    검색결과:
    {text}
    """

    return watson_llm.invoke(prompt).content

In [65]:
import numexpr

@tool
def calculator_tool(expression):
    """
    수학 계산용 Tool
    """
    result = numexpr.evaluate(expression)
    return str(result.item())

calculator_tool.invoke("2+3 * (4-1)")

'11'

In [66]:
from datetime import datetime

@tool
def save_report_tool(content):
    """
    리서치 결과를 Markdown 파일로 저장합니다.
    최종 리포트 작성이 완료되었을 때 사용하세요.
    입력 : Markdown 형식의 리포트 전체 내용
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'research_report_{timestamp}.md'
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)

    return f"리포트 저장 완료: {filename}"

content = """
리서치 결과를 Markdown 파일로 저장합니다.
최종 리포트 작성이 완료되었을 때 사용하세요.
입력 : Markdown 형식의 리포트 전체 내용
"""

save_report_tool.invoke(content)

'리포트 저장 완료: research_report_20260605_154415.md'

In [67]:
tools = [search_tool, trend_analysis_tool, calculator_tool, save_report_tool]

agent = create_agent(
    model=watson_llm,
    tools=tools,
    system_prompt=(
        "You are a research automation assistant. "
        "First use google_search for current information, then use trend_analysis_tool to analyze the search results. "
        "Use calculator_tool only when numeric calculation is needed. "
        "Finally call save_report_tool with the completed Markdown report."
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "AI 반도체 시장 2026년 전망에 대한 리서치 리포트를 작성해주세요. 주제: AI 반도체 시장 2026년 전망. 포함 내용: 주요 기업, 시장 규모, 성장률, 핵심 트렌드",
            }
        ]
    }
)

print_result(result)


========= 0 ==========
HumanMessage
AI 반도체 시장 2026년 전망에 대한 리서치 리포트를 작성해주세요. 주제: AI 반도체 시장 2026년 전망. 포함 내용: 주요 기업, 시장 규모, 성장률, 핵심 트렌드

========= 1 ==========
AIMessage


========= 2 ==========
ToolMessage
The global semiconductor market in 2026 is set to enter a period of high-speed growth fueled by the AI explosion and electrification. The market size is projected to soar from approximately $627 billion in 2024 to between $975 billion and $1 trillion, maintaining a Compound Annual Growth Rate (CAGR) of 8-12%.

========= 3 ==========
AIMessage


========= 4 ==========
ToolMessage
Based on the provided search results, here is a structured analysis of the global semiconductor market:

### 1. Key Trends
- **AI Explosion**: The rapid advancement and integration of Artificial Intelligence (AI) technologies are driving significant demand for specialized semiconductor chips. This includes AI chips that are essential for processing complex AI models.
- **Electrification**: The shift towards el